# Political tweet graph experiments

Upload the original `political_tweets_dataset.zip`. A GPU runtime speeds up the label stage; the scripts also work on CPU. The targets are generated by an NLI teacher, not verified fact-checks.

This notebook is a thin runner for the code in the repository. It tests label filtering, sentence features, larger GraphSAGE models, and a few controls. Runtime results describe whichever machine runs this notebook.

In [ ]:
import os, subprocess, sys
from pathlib import Path

repo = Path('/content/Political-Misinformation-Social-Networks')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/VikiShmiki/Political-Misinformation-Social-Networks.git', str(repo)], check=True)
os.chdir(repo)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

from google.colab import files
uploaded = files.upload()
archives = [name for name in uploaded if name.endswith('.zip')]
assert len(archives) == 1, 'Upload the one tweet archive.'
archive = str(repo / archives[0])

def run(*args):
    subprocess.run([sys.executable, *args], check=True)

run('-m', 'unittest', 'discover', '-s', 'tests', '-v')

In [ ]:
# Both prompt scores are saved after each chunk, so this can be resumed.
run('src/prepare_labels.py', '--archive', archive, '--out', 'work/labels-v2-fp32')
labels = 'work/labels-v2-fp32/labeled_tweets.csv'

import pandas as pd
tweets = pd.read_csv(labels, dtype={'tweet_id': str})
display(tweets.groupby(['eligible', 'accepted', 'target']).size().rename('tweets').to_frame())

In [ ]:
# Full comparison: ten neural configurations, three seeds, common test rows.
run('src/extended_experiments.py', '--labels', labels, '--out', 'results/colab-run')

import json
summary = json.loads(Path('results/colab-run/summary.json').read_text())
display(pd.DataFrame({name: {'binary_macro_f1': value['label_macro_f1']['mean'],
                              'sd': value['label_macro_f1']['sample_std'],
                              'parameters': value['parameters']}
                      for name, value in summary.items()}).T)

In [ ]:
# Run benchmarks after training, with no competing notebook work.
run('src/benchmark_teacher.py', '--archive', archive, '--out', 'results/colab-run/teacher_timing.json')
run('src/build_artifacts.py', '--results', 'results/colab-run', '--labels', labels,
    '--label-manifest', 'work/labels-v2-fp32/label_manifest.json',
    '--predictions', 'work/experiment-cache/colab-run/predictions')
if not Path('work/manual_review.csv').exists():
    run('src/review_queue.py', '--labels', labels)

import shutil
shutil.make_archive('/content/gnn_revision_results', 'zip', 'results/colab-run')
files.download('/content/gnn_revision_results.zip')

## Reading the results

The accepted-label models still get tested on *every eligible test tweet*, including ones the teacher gate rejected. Compare SAGE-small with SAGE-filtered to see the supervision change, then compare filtered/wide/deep for capacity. The MLP and no-edge controls help check whether the graph adds signal.

The first teacher view fixes the target. The second checks consistency. Agreement between those views, or between a student model and its teacher, does not tell us whether a political claim is true. The local review CSV has blank annotation fields for checking that separately.